In [ ]:
import cl
import time
import tables
import numpy as np
import pathlib
import warnings

from common_cl_code.datasets import DATA_BASE_PATH

In [ ]:
baseline_fr_file = DATA_BASE_PATH / 'baseline_fr.npz'
overwrite = True
recording_length_minutes = 30

In [ ]:
timestamp_hz = 25_000

In [ ]:
if not baseline_fr_file.exists() or overwrite:
    import datetime, pytz
    print(datetime.datetime.now(pytz.timezone('US/Eastern')))
    with cl.open() as neurons:
        recording = neurons.record(
            stop_after_seconds=60*recording_length_minutes,
            include_spikes = True,
            include_stims = True,
            include_raw_samples=False,
            include_data_streams=False
        )

        recording.wait_until_stopped()

        attrs = recording.attributes
        recording_path = recording.file['path']
    
    print(f"Recorded {attrs['duration_frames']} frames ({attrs['duration_seconds']} seconds)")
    print(f"into file: {recording.file['path']}")

    del recording
    with tables.open_file(recording_path) as h5_file:
        counts, bins = np.histogram(h5_file.root.spikes.col('channel'), bins = np.arange(65));
    
    baseline_fr = counts / (attrs['duration_frames'] / timestamp_hz)
    
    if not np.allclose(attrs['duration_frames'] / attrs['duration_seconds'], timestamp_hz):
        warnings.warn("Sampling frequency doesn't check out.")
    
    
    np.savez(baseline_fr_file, baseline_fr=baseline_fr, recording_path=recording_path, duration_frames=attrs['duration_frames'], allow_pickle=False)

In [ ]:
file_data = np.load(baseline_fr_file)
baseline_fr = file_data['baseline_fr']
recording_path = pathlib.Path(str(file_data['recording_path']))
duration_frames = file_data['duration_frames']

In [ ]:
import matplotlib.pyplot as plt
plt.plot(baseline_fr, '.')

In [ ]:
plt.matshow(baseline_fr.reshape(8,8).T)

In [ ]:
with tables.open_file(recording_path) as h5_file:
    t = h5_file.root.spikes.col('timestamp')
    c = h5_file.root.spikes.col('channel')
    print(h5_file)

In [ ]:
(t[-1] - t[0])/

In [ ]:
plt.scatter(t/timestamp_hz, c, marker='|', color='k')